# M3: probabilistic layer

A thin driver over a closed loop on synthetic data:

1. A synthetic spectrum with known truth: selection rules, Boltzmann weights, noise, blends, spurious peaks.
2. A rough model fitted to a few low-J levels gives predicted lines with predictive σ.
3. Assignment: χ²-gated, position + intensity likelihood, monotone DP per series, posteriors, ambiguity sets, the Poisson chance null and Benjamini-Hochberg.
4. A fit to the confidently assigned lines, followed by level windows, near-crossing flags and coverage by band.
5. The outlier mixture and level statistics.

The constants are **illustrative** (H2CO-like magnitudes). They are not fitted values for any molecule.

In [ ]:
import math, os
from pathlib import Path
import numpy as np
import riss_drucker as rd
from riss_drucker import assign, posterior, stats
from riss_drucker.rotor import asym_top_labels

WORK = Path(os.environ.get("RISS_DRUCKER_WORK", "/kaggle/working" if Path("/kaggle/working").is_dir() else "."))
TRUE = {"A": 9.405, "B": 1.2953, "C": 1.1342, "DeltaJ": 2.4e-6, "DeltaJK": 9.7e-5, "DeltaK": 4.4e-3,
        "deltaJ": 3.4e-7, "deltaK": 2.7e-5}
MU, T = (0.2, 1.0, 0.3), 150.0
model = rd.compile_asym_rotor(j_max=15, order=4)
peaks, truth_lines = rd.synth.synthesize_spectrum(model, TRUE, MU, T, fwhm=2e-3, snr=3000, spurious_density=0.2, seed=0)
print(len(peaks), "peaks,", sum(p.truth is None for p in peaks), "spurious; band",
      round(peaks[0].position, 2), "-", round(peaks[-1].position, 2), "cm-1")

## Rough prior model → predicted lines with σ → assignment

In [ ]:
lab = asym_top_labels(model)
low_j = rd.synth.synthesize_levels(model, TRUE, noise=1e-4, seed=1, labels=lab, block_filter=lambda b: b["two_j"] <= 12)
prior_fit = rd.fit(model, low_j, rd.synth.perturb_theta(TRUE, 0.01, seed=2))
pred, _ = assign.predict_lines(model, prior_fit.theta, MU, T, covariance=prior_fit.covariance, min_relative_intensity=1e-4)
pred = assign.merge_blends(pred, 1e-3)
res = assign.assign(pred, peaks)
members = lambda s: set(s.split("+")) if s else set()
rows = [r for r in res.lines if r["peak"] is not None]
wrong = sum(1 for r in rows if not members(r["line"]) & members(peaks[r["peak"]].truth))
print(f"assigned {res.n_assigned}/{len(pred)} predicted lines; BH-accepted {res.n_accepted}; ambiguous {res.n_ambiguous}")
print(f"misassignments: actual {wrong}, expected {res.expected_misassignments:.2f}; intensity scale {res.intensity_scale:.3f}")
amb = [r for r in res.lines if r["ambiguous"]][:3]
for r in amb:
    print("  ambiguous:", r["line"][:40], "credible set", r["credible_set"][:3], "posterior %.2f" % r["posterior"])

In [ ]:
obs = assign.line_observations(res, pred, peaks)
fit = rd.fit(model, obs, prior_fit.theta)
print(f"line fit: {len(obs)} lines, converged={fit.converged}, certified={fit.certified}, chi2/dof={fit.reduced_chi2:.2f}")
for n in model.param_names:
    print(f"  {n:8s} {fit.theta[n]:14.8g} +- {fit.sigma[n]:.1e}   pull {(fit.theta[n]-TRUE[n])/fit.sigma[n]:+.2f}")

## Level windows, near crossings, Weyl fence

In [ ]:
w = posterior.level_windows(model, fit, observations=None, labels=lab)
print("levels:", len(w), "| near-crossing:", sum(x["near_crossing"] for x in w),
      "| beyond Weyl fence:", sum(x["beyond_weyl_fence"] for x in w))
worst = sorted(w, key=lambda x: -x["sigma"])[:4]
for x in worst:
    print(f"  {x['label']:>9s}  E = {x['energy']:10.5f} +- {x['sigma']:.1e}   sigma/gap = {x['sigma_over_gap']:.1e}")

## Coverage by band: nominal vs empirical (held-out J parity bands)

In [ ]:
levels = rd.synth.synthesize_levels(model, TRUE, noise=1e-4, seed=3, labels=lab)
rep = posterior.coverage_by_band(model, levels, TRUE, band_of=lambda o: "J even" if int(o.label.split("(")[0]) % 2 == 0 else "J odd")
for band, b in rep["bands"].items():
    print(f"  {band:7s} n={b['n']:4d} empirical {np.round(b['empirical'], 3)} vs nominal {rep['nominal']}")
print("  overall", np.round(rep["overall"]["empirical"], 3), "+-", np.round(rep["overall"]["binomial_se"], 3))

## Outlier mixture

In [ ]:
dirty = [rd.Observation(**{**o.__dict__}) for o in levels]
rng = np.random.default_rng(4)
bad = rng.choice(len(dirty), size=len(dirty) // 20, replace=False)
for i in bad:
    dirty[i].energy += rng.choice([-1, 1]) * rng.uniform(30, 100) * 1e-4
plain = rd.fit(model, dirty, TRUE)
rob = rd.robust.fit_robust(model, dirty, TRUE)
p = np.asarray(rob.outlier_probability)
print(f"plain fit chi2/dof = {plain.reduced_chi2:.1f}; robust outlier fraction = {rob.outlier_fraction:.3f} (true {len(bad)/len(dirty):.3f})")
print(f"injected outliers flagged (p > 0.9): {np.mean(p[bad] > 0.9):.2f}; clean flagged: {np.mean(np.delete(p, bad) > 0.5):.3f}")

## Level statistics reference (⟨r⟩ needs no unfolding)

In [ ]:
rng = np.random.default_rng(0)
for beta, name in [(0, "poisson"), (1, "goe"), (2, "gue"), (4, "gse")]:
    r = np.concatenate([stats.gap_ratios(stats.central(stats.beta_hermite_levels(400, beta, rng))) for _ in range(10)])
    print(f"beta={beta}: <r> = {r.mean():.3f} (reference {stats.GAP_RATIO_REFERENCE[name]:.3f})")

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if plt is not None:
    fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    pos = np.array([p.position for p in peaks]); inten = np.array([p.intensity for p in peaks])
    floor = inten.min() / 3
    ax[0].vlines(pos, floor, inten, color="0.6", lw=0.6, label="observed peaks")
    ok = [r for r in rows if r["bh_accepted"] and r["posterior"] > 0.95]
    ax[0].vlines([r["observed"] for r in ok], floor, [peaks[r["peak"]].intensity for r in ok], color="C0", lw=0.6,
                 label="confident assignment")
    ax[0].set_yscale("log"); ax[0].set_ylim(floor, inten.max() * 2)
    ax[0].set_ylabel("relative intensity"); ax[0].legend(fontsize=8, loc="upper right")
    ax[1].scatter([r["observed"] for r in rows], [r["posterior"] for r in rows], s=5, c=["C3" if r["ambiguous"] else "C0" for r in rows])
    ax[1].set_ylabel("posterior (red: ambiguous)"); ax[1].set_xlabel("wavenumber / cm$^{-1}$")
    fig.tight_layout(); fig.savefig(WORK / "03_assignment.png", dpi=150); print("saved", WORK / "03_assignment.png")